In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

previous_cache = project_root / "data/processed/lab10"
cache_dir = project_root / "data/processed/lab11"
cache_dir.mkdir(parents=True, exist_ok=True)

train_pool = pd.read_parquet(
    previous_cache / "corpus.parquet"
)
dev_queries = pd.read_parquet(
    previous_cache / "queries.parquet"
)

assert train_pool["question_id"].is_unique
assert dev_queries["question_id"].is_unique

assert set(train_pool["question_id"]).isdisjoint(
    set(dev_queries["question_id"])
)
assert set(train_pool["text_key"]).isdisjoint(
    set(dev_queries["text_key"])
)

print(f"Training/reference questions: {len(train_pool):,}")
print(f"Development queries: {len(dev_queries):,}")

Training/reference questions: 1,270
Development queries: 318


In [2]:
def make_training_triplets(
    frame: pd.DataFrame,
    *,
    seed: int = 42,
) -> pd.DataFrame:
    """Create one category-based triplet per question."""
    groups = {
        category: group["question_id"].to_numpy()
        for category, group in frame.groupby(
            "category", sort=True
        )
    }

    if (
        len(groups) < 2
        or any(len(ids) < 2 for ids in groups.values())
    ):
        raise ValueError(
            "Need at least two categories "
            "and two questions per category."
        )

    rng = np.random.default_rng(seed)
    records = []

    for row in frame.itertuples(index=False):
        positive_ids = groups[row.category]
        positive_ids = positive_ids[
            positive_ids != row.question_id
        ]

        other_categories = [
            category
            for category in groups
            if category != row.category
        ]
        negative_category = rng.choice(other_categories)

        records.append({
            "anchor_id": int(row.question_id),
            "positive_id": int(rng.choice(positive_ids)),
            "negative_id": int(
                rng.choice(groups[negative_category])
            ),
        })

    return pd.DataFrame(records)


triplets = make_training_triplets(train_pool)

print(f"Training triplets: {len(triplets):,}")
display(triplets.head())

Training triplets: 1,270


,anchor_id,positive_id,negative_id
0,2794,2796,8649
1,5640,5593,7439
2,2167,2119,8674
3,6346,6459,8774
4,6428,6357,6023


In [3]:
lookup = train_pool.set_index("question_id")

for role in ["anchor", "positive", "negative"]:
    assert triplets[f"{role}_id"].isin(lookup.index).all()

assert triplets["anchor_id"].is_unique
assert triplets["anchor_id"].ne(
    triplets["positive_id"]
).all()

anchor_categories = triplets["anchor_id"].map(
    lookup["category"]
)
positive_categories = triplets["positive_id"].map(
    lookup["category"]
)
negative_categories = triplets["negative_id"].map(
    lookup["category"]
)

assert anchor_categories.eq(positive_categories).all()
assert anchor_categories.ne(negative_categories).all()

print("Triplet checks passed.")

Triplet checks passed.


In [6]:
inspection = triplets.head(3).copy()
inspection

,anchor_id,positive_id,negative_id
0,2794,2796,8649
1,5640,5593,7439
2,2167,2119,8674


In [8]:
inspection = triplets.head(3).copy()

for role in ["anchor", "positive", "negative"]:
    inspection[f"{role}_text"] = inspection[
        f"{role}_id"
    ].map(lookup["text"])

    inspection[f"{role}_category"] = inspection[
        f"{role}_id"
    ].map(lookup["category"])

for row in inspection.itertuples(index=False):
    print("ANCHOR:", row.anchor_text)
    print("Category:", row.anchor_category)

    print("POSITIVE:", row.positive_text)
    print("Category:", row.positive_category)

    print("NEGATIVE:", row.negative_text)
    print("Category:", row.negative_category)
    print()

ANCHOR: How long will my transaction take?
Category: transfer_not_received_by_recipient
POSITIVE: i accidentally entered in my retailers information, what should i do next
Category: transfer_not_received_by_recipient
NEGATIVE: Where is my money that I transfered?!
Category: balance_not_updated_after_bank_transfer

ANCHOR: I can't seem to figure out why my transfers are getting declined.
Category: declined_transfer
POSITIVE: What does the decline message mean
Category: declined_transfer
NEGATIVE: What would cause a transfer failure?
Category: failed_transfer

ANCHOR: I need to cancel my recent transfer as soon as possible. I made an error there. Please help before it goes through.
Category: cancel_transfer
POSITIVE: I wanna revert a transaction i did this morning.
Category: cancel_transfer
NEGATIVE: I transfered my balance a while back.  Why doesn't my account reflect this?
Category: balance_not_updated_after_bank_transfer



In [10]:
train_pool.to_parquet(
    cache_dir / "train_pool.parquet",
    index=False,
)
dev_queries.to_parquet(
    cache_dir / "dev_queries.parquet",
    index=False,
)
triplets.to_parquet(
    cache_dir / "training_triplets.parquet",
    index=False,
)

print("Lab 11 training examples saved.")

Lab 11 training examples saved.
